# Step 1 — GTFS TOD Stop Classification

Reads the regional GTFS feed and the Caltrans HQTS service, classifies
TOD-applicable stops, assigns Tier 1 / Tier 2 designations, and exports
`stops`, `stations`, and `access_points` layers to the shared GeoPackage.

> **Pipeline order:** Run this notebook first, then proceed to
> `2_tod_stop_and_access_assignment.ipynb`.
> All data source paths are defined in `config.py`.


In [1]:
import gtfs_kit as gk
import pandas as pd
import geopandas as gpd
from config import *

In [2]:
# Read GTFS feed and geometrize stops
feed = gk.read_feed(GTFS_ZIP, dist_units='mi')
stops_gdf = gk.geometrize_stops(feed.stops)

# Load major BRT stops from Caltrans HQTS service (filtered to major_stop_brt)
hqts_gdf = gpd.read_file(HQTS_URL)
print(f"Loaded {len(hqts_gdf)} major BRT stops from HQTS data")


Loaded 600 major BRT stops from HQTS data


In [3]:
# Enrich stops with agency and route information
# Join through GTFS relational structure: stops → stop_times → trips → routes → agency

# Get unique stop-route relationships
stop_routes = (
    feed.stop_times[["stop_id", "trip_id"]]
    .merge(feed.trips[["trip_id", "route_id"]], on="trip_id")
    .drop_duplicates(["stop_id", "route_id"])
)

# Add route details
stop_routes = stop_routes.merge(
    feed.routes[["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]],
    on="route_id",
)

# Add agency details
stop_routes = stop_routes.merge(
    feed.agency[["agency_id", "agency_name"]], on="agency_id", how="left"
)

# Filter to only relevant transit agencies (defined in config.py)
stop_routes = stop_routes[stop_routes['agency_id'].isin(RELEVANT_AGENCIES)]

# Aggregate by stop_id to get lists of agencies and routes per stop
stop_info = (
    stop_routes.groupby("stop_id")
    .agg(
        {
            "agency_name": lambda x: ", ".join(sorted(set(x.dropna()))),
            "agency_id": lambda x: ", ".join(sorted(set(x.dropna()))),
            "route_short_name": lambda x: ", ".join(sorted(set(x.dropna()))),
            "route_long_name": lambda x: ", ".join(sorted(set(x.dropna()))),
            "route_type": lambda x: ", ".join(map(str, sorted(set(x.dropna())))),
        }
    )
    .reset_index()
)

# Merge with stops_gdf
stops_gdf = stops_gdf.merge(stop_info, on="stop_id", how="left")


In [4]:
# Separate stations, stops, and access points based on GTFS hierarchy

# Create stations dataset (location_type = 1)
# These are parent structures that contain stops
stations_gdf = stops_gdf[stops_gdf["location_type"] == 1][
    ["stop_id", "stop_name", "location_type", "geometry"]
].copy()
# Rename stop_id to station_id for clarity (this will be the primary key for stations)
stations_gdf = stations_gdf.rename(columns={"stop_id": "station_id", "stop_name": "station_name"})

# Create access point dataset (location_type = 2 or 3)
access_pts_gdf = stops_gdf[stops_gdf["location_type"].isin([2, 3])].copy()

# rename columns for clarity
access_pts_gdf = access_pts_gdf.rename(
    columns={
        "stop_id": "access_id",
        "stop_name": "access_point_name",
        "parent_station": "station_id",
    }
)

# drop columns other than access_id, access_point_name, station_id, location_type, geometry
cols_to_keep = ["access_id", "access_point_name", "station_id", "location_type", "geometry"]
access_pts_gdf = access_pts_gdf[cols_to_keep]

# Create stops dataset (location_type = 0 or blank)
# Filter to only stops that are actually served by routes (have route info)
stops_gdf = stops_gdf[
    (stops_gdf["location_type"].fillna(0) == 0) & (stops_gdf["route_short_name"].notna())
].copy()

# Rename parent_station to station_id for clarity (this will be the foreign key to stations)
stops_gdf = stops_gdf.rename(columns={"parent_station": "station_id"})

# Clean up stops_gdf: remove unnecessary columns
cols_to_drop = [
    "stop_code",
    "stop_desc",
    "zone_id",
    "stop_url",
    "tts_stop_name",
    "platformcode",
    "stop_timezone",
    "wheelchair_boarding",
    "route_long_name",
]
stops_gdf = stops_gdf.drop(columns=[col for col in cols_to_drop if col in stops_gdf.columns])

# Move geometry to the end
cols = [col for col in stops_gdf.columns if col != "geometry"] + ["geometry"]
stops_gdf = stops_gdf[cols]

print(f"Total records in original data: {len(stops_gdf) + len(stations_gdf)}")
print(f"Stations (location_type=1): {len(stations_gdf)}")
print(f"Stops (location_type=0, served by routes): {len(stops_gdf)}")
print(f"Stops with station_id: {stops_gdf['station_id'].notna().sum()}")
print(f"Access points (location_type=2 or 3): {len(access_pts_gdf)}")
print(f"\nColumns in stops dataset: {list(stops_gdf.columns)}")
print(f"Columns in stations dataset: {list(stations_gdf.columns)}")
print(f"Columns in access points dataset (location_type=2 or 3): {list(access_pts_gdf.columns)}")
# Show the relationship structure
print("\nData model relationships:")
print("stops.stop_id (PK) → unique stop identifier")
print("stops.station_id (FK) → references stations.station_id")
print("stations.station_id (PK) → unique station identifier")
print("access_pts.access_id (PK) → unique access point identifier")
print("access_pts.station_id (FK) → references stations.station_id")

Total records in original data: 11658
Stations (location_type=1): 301
Stops (location_type=0, served by routes): 11357
Stops with station_id: 423
Access points (location_type=2 or 3): 1236

Columns in stops dataset: ['stop_id', 'stop_name', 'platform_code', 'location_type', 'station_id', 'level_id', 'agency_name', 'agency_id', 'route_short_name', 'route_type', 'geometry']
Columns in stations dataset: ['station_id', 'station_name', 'location_type', 'geometry']
Columns in access points dataset (location_type=2 or 3): ['access_id', 'access_point_name', 'station_id', 'location_type', 'geometry']

Data model relationships:
stops.stop_id (PK) → unique stop identifier
stops.station_id (FK) → references stations.station_id
stations.station_id (PK) → unique station identifier
access_pts.access_id (PK) → unique access point identifier
access_pts.station_id (FK) → references stations.station_id


In [5]:
# View sample of stations
print("Sample stations:")
stations_gdf.head(10)

Sample stations:


,station_id,station_name,location_type,geometry
0,mtc:powell,Powell,1,POINT (-122.40737 37.78459)
1,mtc:fruitvale,Fruitvale,1,POINT (-122.22412 37.77486)
2,mtc:warm-springs-south-fremont-bart,Warm Springs South Fremont BART,1,POINT (-121.93925 37.50199)
3,mtc:caltrain-4th-&-king,Caltrain 4th & King,1,POINT (-122.39487 37.77649)
4,mtc:el-cerrito-del-norte-bart,El Cerrito Del Norte BART,1,POINT (-122.31699 37.92529)
5,mtc:walnut-creek-bart,Walnut Creek BART,1,POINT (-122.06734 37.9058)
6,mtc:richmond-bart-amtrak,Richmond BART/Amtrak,1,POINT (-122.35314 37.93684)
7,mtc:santa-clara-caltrain,Santa Clara Caltrain,1,POINT (-121.93555 37.35313)
8,mtc:san-francisco-ferry-terminal,San Francisco Ferry Terminal,1,POINT (-122.39341 37.79553)
9,mtc:macarthur-bart,MacArthur BART,1,POINT (-122.26706 37.82909)


In [6]:
# View sample of served stops with route/agency information
print("Sample stops with route/agency info:")
stops_gdf.head(10)

Sample stops with route/agency info:


,stop_id,stop_name,platform_code,location_type,station_id,level_id,agency_name,agency_id,route_short_name,route_type,geometry
306,16995,Metro Powell Station/Outbound,<NA>,0,mtc:powell,mtc:powell-muni-platform,San Francisco Municipal Transportation Agency,SF,"J, K, L, M, N",0,POINT (-122.40782 37.7843)
307,15417,Metro Powell Station/Downtown,<NA>,0,mtc:powell,mtc:powell-muni-platform,San Francisco Municipal Transportation Agency,SF,"J, K, L, M, N",0,POINT (-122.40709 37.78465)
308,51333,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"19, 51A, 851",3,POINT (-122.22457 37.775)
309,59000,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"30, 31, 639",3,POINT (-122.22474 37.77511)
310,55550,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"14, 54",3,POINT (-122.22489 37.7752)
311,55355,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,O,3,POINT (-122.22521 37.77523)
312,53111,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"19, 648, 654, 655",3,POINT (-122.22537 37.77533)
313,55533,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"30, 31, 639",3,POINT (-122.22511 37.77534)
314,55522,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"62, 703",3,POINT (-122.22528 37.77543)
315,55886,Fruitvale BART,<NA>,0,mtc:fruitvale,mtc:fruitvale-street-level,AC TRANSIT,AC,"14, 54, 62",3,POINT (-122.22546 37.77552)


In [7]:
# View sample of access points
print("Sample access points:")
access_pts_gdf.head(10)

Sample access points:


,access_id,access_point_name,station_id,location_type,geometry
301,mtc:powell_n1_concourse,<NA>,mtc:powell,3,POINT (-122.40769 37.7842)
302,mtc:1586751243014,B1 Stairs (Concourse Level),mtc:powell,3,POINT (-122.40821 37.78379)
303,mtc:powell_n6_concourse_muni,Concourse,mtc:powell,3,POINT (-122.40629 37.78542)
304,mtc:powell_n8_concourse_elevator,Platform Elevator (Concourse Level),mtc:powell,3,POINT (-122.40625 37.78545)
305,mtc:1586757225033,BART Platform,mtc:powell,3,POINT (-122.4069 37.78489)
316,mtc:fruitvale_bus_bays,Fruitvale Bus Bays,mtc:fruitvale,3,POINT (-122.22474 37.77521)
317,mtc:fruitvale_fare_area,Fruitvale Fare Area,mtc:fruitvale,3,POINT (-122.22431 37.77498)
318,mtc:fruitvale_bart_north,Fruitvale BART Platform (N),mtc:fruitvale,3,POINT (-122.22409 37.77492)
319,mtc:fruitvale_bart_south,Fruitvale BART Platform (S),mtc:fruitvale,3,POINT (-122.22417 37.7748)
322,mtc:1590030012044,<NA>,mtc:caltrain-4th-&-king,3,POINT (-122.3949 37.77669)


In [8]:
# Flag transit-oriented development (TOD) applicable stops and assign tiers

# Get major BRT stop IDs (already filtered to major_stop_brt in API call)
major_brt_stops = hqts_gdf['stop_id'].dropna().tolist()
print(f"Found {len(major_brt_stops)} major BRT stops in HQTS data")

# Initialize TOD columns
stops_gdf['tod_stop'] = False
stops_gdf['tod_tier'] = None

# Helper function to check if route_type contains specific values
def has_route_type(route_type_str, target_type):
    """Check if comma-separated route_type string contains target type"""
    if pd.isna(route_type_str):
        return False
    return str(target_type) in str(route_type_str).split(', ')

def has_agency(agency_id_str, target_agency):
    """Check if comma-separated agency_id string contains target agency"""
    if pd.isna(agency_id_str):
        return False
    return target_agency in str(agency_id_str).split(', ')

# Step 1: Flag TOD applicable stops
# TOD applicable if:
# - route_type contains 0 (Light Rail) OR 1 (Subway/Metro) 
# - OR agency_id contains 'CT' (Caltrain)
# - OR stop is in HQTS major BRT stops

tod_mask = (
    stops_gdf['route_type'].apply(lambda x: has_route_type(x, 0)) |  # Light Rail
    stops_gdf['route_type'].apply(lambda x: has_route_type(x, 1)) |  # Subway/Metro
    stops_gdf['agency_id'].apply(lambda x: has_agency(x, 'CT')) |    # Caltrain
    stops_gdf['stop_id'].isin(major_brt_stops)                       # Major BRT stops
)

# Apply TOD flag
stops_gdf.loc[tod_mask, 'tod_stop'] = True

# Step 2: Apply exceptions
# TODO: Implement exceptions when geographic data is available
# - Exclude Caltrain stops south of Tamien Station
# - Exclude BART stops in Contra Costa County

# Step 3: Assign tiers for TOD applicable stops
# Tier 1: route_type contains 1 (Subway/Metro) OR agency_id contains 'CT' (Caltrain)
tier_1_mask = (
    stops_gdf['tod_stop'] &
    (stops_gdf['route_type'].apply(lambda x: has_route_type(x, 1)) |  # Subway/Metro 
     stops_gdf['agency_id'].apply(lambda x: has_agency(x, 'CT')))     # Caltrain
)

# Tier 2: route_type contains 0 (Light Rail) OR stop is in HQTS BRT stops
tier_2_mask = (
    stops_gdf['tod_stop'] &
    (stops_gdf['route_type'].apply(lambda x: has_route_type(x, 0)) |  # Light Rail
     stops_gdf['stop_id'].isin(major_brt_stops))                      # Major BRT stops
)

# Assign tiers (Tier 1 takes precedence over Tier 2)
stops_gdf.loc[tier_1_mask, 'tod_tier'] = 'Tier 1'
stops_gdf.loc[tier_2_mask & ~tier_1_mask, 'tod_tier'] = 'Tier 2'

# Summary statistics
print("\nTOD Stop Classification Results:")
print(f"Total stops: {len(stops_gdf)}")
print(f"TOD applicable stops: {stops_gdf['tod_stop'].sum()}")
print(f"Tier 1 stops: {(stops_gdf['tod_tier'] == 'Tier 1').sum()}")
print(f"Tier 2 stops: {(stops_gdf['tod_tier'] == 'Tier 2').sum()}")

# Show breakdown by agency
print("\nTOD stops by agency:")
tod_stops = stops_gdf[stops_gdf['tod_stop'] == True]
if len(tod_stops) > 0:
    print(tod_stops.groupby(['agency_name', 'tod_tier']).size().unstack(fill_value=0))
else:
    print("No TOD stops found")

Found 600 major BRT stops in HQTS data

TOD Stop Classification Results:
Total stops: 11357
TOD applicable stops: 779
Tier 1 stops: 163
Tier 2 stops: 616

TOD stops by agency:
tod_tier                                       Tier 1  Tier 2
agency_name                                                  
AC TRANSIT                                          0      66
Bay Area Rapid Transit                            103       0
Caltrain                                           60       0
San Francisco Municipal Transportation Agency       0     429
VTA                                                 0     121


In [9]:
# Export datasets to the shared GeoPackage
print("Exporting datasets to GeoPackage...")
print(f"Output: {TOD_DATABASE_GPKG}")

stops_gdf.to_file(TOD_DATABASE_GPKG, layer=GPKG_STOPS_LAYER, driver='GPKG', mode='w')
print(f"Exported {len(stops_gdf)} stops to '{GPKG_STOPS_LAYER}' layer")

stations_gdf.to_file(TOD_DATABASE_GPKG, layer=GPKG_STATIONS_LAYER, driver='GPKG', mode='w')
print(f"Exported {len(stations_gdf)} stations to '{GPKG_STATIONS_LAYER}' layer")

access_pts_gdf.to_file(TOD_DATABASE_GPKG, layer=GPKG_ACCESS_PTS_LAYER, driver='GPKG', mode='w')
print(f"Exported {len(access_pts_gdf)} access points to '{GPKG_ACCESS_PTS_LAYER}' layer")

print(f"\nGeoPackage updated: {TOD_DATABASE_GPKG}")
print(f"\nData model:")
print("• stops.stop_id (PK) → unique stop identifier")
print("• stops.station_id (FK) → references stations.station_id")
print("• stations.station_id (PK) → unique station identifier")
print("• access_pts.access_id (PK) → unique access point identifier")
print("• access_pts.station_id (FK) → references stations.station_id")


Exporting datasets to GeoPackage...
Output: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg
Exported 11357 stops to 'stops' layer
Exported 301 stations to 'stations' layer
Exported 1236 access points to 'access_points' layer

✅ GeoPackage updated: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg

Data model:
• stops.stop_id (PK) → unique stop identifier
• stops.station_id (FK) → references stations.station_id
• stations.station_id (PK) → unique station identifier
• access_pts.access_id (PK) → unique access point identifier
• access_pts.station_id (FK) → references stations.station_id
